# Final Release Results Analysis

**Objective.** Reproduce the final paper-facing training, runtime, utility, and freeze tables from locked CPU release artifacts.

**Run mode.** Analysis only. This notebook reads locked ORIUS artifacts
and does not retrain models, rewrite release manifests, or mutate runtime traces.

In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "reports").exists():
    ROOT = Path.cwd().parent
PUBLICATION = ROOT / "reports" / "publication"
SPLIT_ROOT = ROOT / "reports" / "split_training"
RELEASE_ID = (SPLIT_ROOT / "latest_release_id.txt").read_text().strip()
FREEZE = ROOT / "reports" / "predeployment_freeze" / RELEASE_ID

def read_csv(relpath: str) -> pd.DataFrame:
    path = ROOT / relpath
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

def read_json(relpath: str) -> dict:
    path = ROOT / relpath
    if not path.exists():
        raise FileNotFoundError(path)
    return json.loads(path.read_text())

def display_path(relpath: str) -> None:
    path = ROOT / relpath
    print(f"{relpath}: {'exists' if path.exists() else 'missing'}")

## Final release tables

In [ ]:
training = read_csv("reports/publication/final_training_quality_for_paper.csv")
runtime = read_csv("reports/publication/final_runtime_safety_for_paper.csv")
freeze = read_csv("reports/publication/final_freeze_validation_for_paper.csv")
utility = read_csv("reports/publication/utility_preserving_safety_scorecard.csv")

print("release:", RELEASE_ID)
display(training)
display(runtime)
display(utility[["domain", "safety_reference_controller", "excess_tsvr_over_safety_reference", "utility_gain_over_safety_reference", "utility_preserving_safety_gate"]])
display(freeze)

## Runtime TSVR reduction

In [ ]:
ax = runtime.plot(
    x="domain",
    y=["baseline_tsvr", "orius_tsvr"],
    kind="bar",
    figsize=(9, 4),
    title="Final claim-governing TSVR",
)
ax.set_ylabel("TSVR")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()

In [ ]:
assert (runtime["orius_tsvr"] <= runtime["baseline_tsvr"]).all()
assert (freeze["pass"].astype(str).str.lower() == "yes").all()
assert (utility["utility_preserving_safety_gate"].astype(str).str.lower() == "true").all()